# 32 · CXR-LT — baseline **CXR-only** + so ba trục (Stage 3b/c/d)

Probe tuyến tính trên embedding **đông cứng** của `31`. Đây là sản phẩm chính của kế hoạch: ba mốc
gốc **EHR-only · CXR-only · EHR+CXR**, cộng hai phép so nội bộ.

## Bốn câu notebook này trả lời

| | phép so | trả lời |
|---|---|---|
| **1** | `crop518` vs `lb518` | **FOV có đáng không** — 71.5% → 100% khung, đổi lấy độ phân giải 4.1% → 2.9% pixel gốc |
| **2** | `cls` vs `patch` vs `cls+patch` | biểu diễn nào mạnh hơn (`31` đo được `cos ≈ 0.1–0.15`, tức **gần trực giao** — nối lại có thể hơn cả hai) |
| **3** | mức **ảnh** vs mức **study** | gộp `0.7·frontal + 0.3·lateral` có giúp không |
| **4** | CXR-only vs EHR-only vs **EHR+CXR** | ba mốc gốc |

## Hai mức chấm điểm, và vì sao cần cả hai

CXR-LT chấm ở **mức ảnh** — mỗi `dicom_id` một dòng dự đoán. Đó là con số **so được với
leaderboard** (vô địch Task 1: **0.281 mAP**).

Còn thiết kế của ta gộp ở **mức study** với `0.7·frontal + 0.3·lateral` — có tiền lệ ở CXR-LT 2024
(Team F dùng đúng trung bình có trọng số thiên frontal). Con số đó phản ánh pipeline của ta nhưng
**không so được** với leaderboard.

Nên báo cáo cả hai, đừng trộn.

## Hiệu chỉnh kỳ vọng trước khi đọc số

Đây là **đặc trưng đông cứng + probe tuyến tính** — là **sàn**, không phải bài dự thi. Vô địch đạt
0.281 mAP bằng fine-tune toàn mạng + ensemble đa độ phân giải + weighted ASL. Ở đây rơi vào khoảng
**0.15–0.22 mAP** là bình thường. Giá trị của nó là làm mốc để mọi thứ thêm vào sau này đo được.

Và nhớ: hạng 2–4 của challenge cách nhau **0.004**, nên **chênh lệch dưới 0.005 là nhiễu**.

**GPU (hoặc CPU cũng chạy được).** ~20–40 phút. Cần `31` đã chạy ít nhất một arm.

## C0 · Config + nạp nhãn, view, split

In [ ]:
import os, json, time, itertools
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'

OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RES_DIR = os.path.join(OUT_DIR, 'stage3_baselines'); os.makedirs(RES_DIR, exist_ok=True)
EHR_DIR = os.path.join(OUT_DIR, 'stage2_ehr_only')

W_FRONTAL, W_LATERAL = 0.7, 0.3     # trong so gop view, khoa tu ke hoach
SEED = 42

INV   = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']
TIER_OF = {l: t for t in ('head', 'medium', 'tail') for l in INV['tier'][t]}

ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'),
                     columns=['dicom_id'] + LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
assert (Y.dicom_id.to_numpy() == ORDER.dicom_id.to_numpy()).all()
YM = Y[LAB40].to_numpy('float32')            # nhan muc ANH, cung thu tu voi embedding

# view: uu tien ban da giai o 28d/28e, khong thi dung cot goc
P_VIEW = os.path.join(OUT_DIR, 'cxrlt_view_resolved.parquet')
if os.path.exists(P_VIEW):
    V = (pd.read_parquet(P_VIEW)[['dicom_id', 'view_group', 'use']]
           .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
    print(f'view: tu cxrlt_view_resolved.parquet | dung duoc {V.use.mean():.2%}')
else:
    V = ORDER[['dicom_id', 'view_group']].copy(); V['use'] = V.view_group != 'unknown'
    print(f'view: tu cxrlt_index.parquet (28d/28e chua chay) | dung duoc {V.use.mean():.2%}')

SPLIT = ORDER.split.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'
ARMS = [a for a in ('crop518', 'lb518', 'nosq518', 'lb1022')
        if os.path.exists(os.path.join(EMB_DIR, f'emb_{a}_cls.npy'))]
assert ARMS, 'chua co embedding nao -> chay 31 truoc'
print(f'\n{len(ORDER):,} anh | train {TR.sum():,} | val {VA.sum():,}')
print(f'arm da co: {ARMS}')

## C1 · Probe tuyến tính — **một lớp, 40 đầu ra**

Không khớp 40 hồi quy riêng: một `Linear(D → 40)` với `BCEWithLogitsLoss` vừa nhanh hơn nhiều vừa
đúng nghĩa *linear probe* chuẩn.

Chuẩn hoá đặc trưng bằng thống kê **của tập train** — nếu tính trên toàn bộ thì val rò rỉ vào
phép chuẩn hoá.

In [ ]:
import torch, torch.nn as nn
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'thiet bi: {dev}')

def fit_probe(Xtr, ytr, Xva, epochs=30, lr=3e-3, wd=1e-4, bs=4096, verbose=False):
    '''Linear(D->C) + BCE. Tra ve xac suat tren Xva.'''
    torch.manual_seed(SEED)
    mu, sd = Xtr.mean(0, keepdims=True), Xtr.std(0, keepdims=True) + 1e-6   # CHI train
    xt = torch.from_numpy((Xtr - mu) / sd).to(dev)
    yt = torch.from_numpy(ytr).to(dev)
    xv = torch.from_numpy((Xva - mu) / sd).to(dev)

    m   = nn.Linear(xt.shape[1], yt.shape[1]).to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=wd)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    lossf = nn.BCEWithLogitsLoss()
    n = len(xt)
    for ep in range(epochs):
        perm = torch.randperm(n, device=dev)
        tot = 0.0
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            opt.zero_grad(set_to_none=True)
            l = lossf(m(xt[idx]), yt[idx]); l.backward(); opt.step()
            tot += l.item() * len(idx)
        sch.step()
        if verbose and (ep+1) % 10 == 0: print(f'   ep{ep+1:>3} loss {tot/n:.4f}')
    with torch.no_grad():
        return torch.sigmoid(m(xv)).cpu().numpy()

def feats(arm, kind):
    '''kind: cls | patch | both'''
    c = np.load(os.path.join(EMB_DIR, f'emb_{arm}_cls.npy'),   mmap_mode='r')
    p = np.load(os.path.join(EMB_DIR, f'emb_{arm}_patch.npy'), mmap_mode='r')
    if kind == 'cls':   return np.asarray(c)
    if kind == 'patch': return np.asarray(p)
    return np.concatenate([np.asarray(c), np.asarray(p)], 1)
print('san sang')

## C2 · Chấm điểm ở **hai mức**

* **mức ảnh** — mỗi `dicom_id` một dự đoán. So được với leaderboard.
* **mức study** — gộp `0.7·frontal + 0.3·lateral` trên **dự đoán** (không phải trên đặc trưng),
  giống cách Team F của CXR-LT 2024 làm. Study nào thiếu một view thì view còn lại nhận trọng số 1.

Ảnh có `use=False` (không xác định được view) **vẫn được chấm ở mức ảnh** nhưng **không tham gia
gộp study** — chúng không có trọng số hợp lệ.

In [ ]:
from sklearn.metrics import average_precision_score

_sid_va = ORDER.study_id.to_numpy()[VA]
_uid, _inv = np.unique(_sid_va, return_inverse=True)
_isF = (V.view_group.to_numpy()[VA] == 'frontal') & V.use.to_numpy()[VA]
_isL = (V.view_group.to_numpy()[VA] == 'lateral') & V.use.to_numpy()[VA]

def pool_study(p_img):
    '''[n_img_val, C] -> [n_study_val, C] bang 0.7 frontal + 0.3 lateral.'''
    C = p_img.shape[1]; out = np.zeros((len(_uid), C), 'float64'); wsum = np.zeros(len(_uid))
    for mask, w in ((_isF, W_FRONTAL), (_isL, W_LATERAL)):
        if not mask.any(): continue
        cnt = np.bincount(_inv[mask], minlength=len(_uid))
        acc = np.stack([np.bincount(_inv[mask], weights=p_img[mask, j].astype('float64'),
                                    minlength=len(_uid)) for j in range(C)], 1)
        has = cnt > 0
        out[has] += w * acc[has] / cnt[has, None]; wsum[has] += w
    ok = wsum > 0
    out[ok] /= wsum[ok, None]
    return out, ok

# nhan muc study (da kiem hang so trong study o 28 C4)
_Y_study = np.zeros((len(_uid), len(LAB40)), 'float32')
_first = np.zeros(len(_uid), int); _first[_inv[::-1]] = np.arange(VA.sum())[::-1]
_Y_study = YM[VA][_first]

def score(p_img):
    '''-> (ap_anh, ap_study) moi cai la vector do dai C.'''
    ya = YM[VA]
    ap_i = np.array([average_precision_score(ya[:, j], p_img[:, j]) for j in range(ya.shape[1])])
    ps, ok = pool_study(p_img)
    ap_s = np.array([average_precision_score(_Y_study[ok][:, j], ps[ok][:, j])
                     for j in range(ya.shape[1])])
    return ap_i, ap_s

print(f'val: {VA.sum():,} anh -> {len(_uid):,} study')
print(f'  co frontal {_isF.sum():,} | co lateral {_isL.sum():,} | '
      f'khong dung duoc {int((~V.use.to_numpy()[VA]).sum()):,}')

## C3 · Khớp mọi tổ hợp `(arm, biểu diễn)`

In [ ]:
P_CSV = os.path.join(RES_DIR, 'cxr_only.csv')
KINDS = ['cls', 'patch', 'both']
done = set()
if os.path.exists(P_CSV):
    _d = pd.read_csv(P_CSV); done = set(zip(_d.arm, _d.kind))
    print(f'resume: da co {len(done)} to hop')

PRED = {}
for arm, kind in itertools.product(ARMS, KINDS):
    f = os.path.join(RES_DIR, f'p_{arm}_{kind}.npy')
    if (arm, kind) in done and os.path.exists(f):
        PRED[(arm, kind)] = np.load(f); continue
    t0 = time.time()
    X = feats(arm, kind)
    p = fit_probe(X[TR], YM[TR], X[VA])
    del X
    np.save(f, p.astype('float32')); PRED[(arm, kind)] = p
    ap_i, ap_s = score(p)
    pd.DataFrame([dict(arm=arm, kind=kind, mAP_anh=ap_i.mean(), mAP_study=ap_s.mean(),
                       **{f'ap_{l}': v for l, v in zip(LAB40, ap_i)})]
                 ).to_csv(P_CSV, mode='a', index=False, header=not os.path.exists(P_CSV))
    print(f'  {arm:<8} {kind:<5} mAP anh {ap_i.mean():.4f} | study {ap_s.mean():.4f} '
          f'| {time.time()-t0:.0f}s')
print('xong')

## C4 · Bảng chính — CXR-only

In [ ]:
R = pd.read_csv(P_CSV).drop_duplicates(['arm', 'kind'], keep='last')
piv = R.pivot(index='arm', columns='kind', values='mAP_anh').reindex(columns=KINDS)
pvs = R.pivot(index='arm', columns='kind', values='mAP_study').reindex(columns=KINDS)
print('=== mAP muc ANH (so duoc voi leaderboard) ===')
print(piv.to_string(float_format=lambda v: f'{v:.4f}'))
print('\n=== mAP muc STUDY (0.7 frontal + 0.3 lateral) ===')
print(pvs.to_string(float_format=lambda v: f'{v:.4f}'))
print('\n=== gop study loi/hai bao nhieu ===')
print((pvs - piv).to_string(float_format=lambda v: f'{v:+.4f}'))

best = R.loc[R.mAP_anh.idxmax()]
print(f'\ntot nhat: {best.arm} / {best.kind} — mAP anh {best.mAP_anh:.4f}, study {best.mAP_study:.4f}')

apc = [c for c in R.columns if c.startswith('ap_')]
B = R[R.arm.eq(best.arm) & R.kind.eq(best.kind)][apc].iloc[0]
T = pd.DataFrame(dict(label=[c[3:] for c in apc], ap=B.to_numpy('float64')))
T['tier'] = T.label.map(TIER_OF)
T['n_pos_val'] = [int(YM[VA][:, LAB40.index(l)].sum()) for l in T.label]
print('\n=== theo TIER (arm tot nhat) ===')
g = T.groupby('tier').agg(n_nhan=('ap', 'size'), mAP=('ap', 'mean'),
                          n_pos_min=('n_pos_val', 'min'))
print(g.reindex(['head', 'medium', 'tail']).to_string(float_format=lambda v: f'{v:.4f}'))
print(f'''
MOC DOI CHIEU — CXR-LT 2024 Task 1 (fine-tune + ensemble, KHONG phai probe dong cung):
  vo dich 0.281 mAP | head 0.567 / medium 0.263 / tail 0.136
  Hang 2-4 cach nhau 0.004 -> chenh lech duoi 0.005 la nhieu.''')

## C5 · **FOV có đáng không** — `crop518` vs `lb518`, tách theo nhãn

Đây là phép so `28c` sinh ra để trả lời. Dự đoán trước khi nhìn số: letterbox **thắng** ở nhóm
ngoại vi (`Pleural Effusion` ở góc sườn hoành, `Pneumothorax` ở đỉnh phổi, `*Fracture` ở sườn bên)
và **thua** ở nhóm cần chi tiết (`Nodule`, `Lung Lesion`, `Granuloma`), vì nó đổi độ phân giải lấy
khung hình.

Nếu bảng ra đúng hình dạng đó thì kết luận rất chắc. Nếu chênh lệch rải rác không theo giải phẫu
thì đó là nhiễu, và macro gộp sẽ che mất điều đó — nên **phải xem theo nhãn**.

In [ ]:
# Cong chan: `R` co thieu dong khong. Thieu xay ra khi mot arm vua bi vo hieu hoa de
# fit lai ma C3 chua chay — truoc day cho ra IndexError kho doc thay vi noi ro thieu gi.
_have = set(zip(R.arm, R.kind))
_need = {('crop518', best.kind), ('lb518', best.kind)}
if _need <= _have:
    kind = best.kind
    a = R[R.arm.eq('crop518') & R.kind.eq(kind)][apc].iloc[0].to_numpy('float64')
    b = R[R.arm.eq('lb518')   & R.kind.eq(kind)][apc].iloc[0].to_numpy('float64')
    F = pd.DataFrame(dict(label=[c[3:] for c in apc], crop518=a, lb518=b))
    F['delta'] = F.lb518 - F.crop518
    F['tier'] = F.label.map(TIER_OF)
    F['n_pos_val'] = [int(YM[VA][:, LAB40.index(l)].sum()) for l in F.label]

    NGOAI_VI = ['Pleural Effusion', 'Pneumothorax', 'Fracture', 'Rib Fracture',
                'Clavicle Fracture', 'Pleural Thickening', 'Pleural Other',
                'Subcutaneous Emphysema', 'Hydropneumothorax']
    CHI_TIET = ['Nodule', 'Lung Lesion', 'Granuloma', 'Calcification of the Aorta', 'Mass']
    F['nhom'] = np.where(F.label.isin(NGOAI_VI), 'ngoai vi',
                 np.where(F.label.isin(CHI_TIET), 'chi tiet', 'khac'))

    print(f'=== lb518 - crop518, bieu dien `{kind}` (mAP anh) ===')
    print(F.sort_values('delta', ascending=False)
           .to_string(index=False, float_format=lambda v: f'{v:.4f}'))
    print('\n=== theo NHOM GIA THUYET ===')
    print(F.groupby('nhom').agg(n=('delta', 'size'), delta_tb=('delta', 'mean'),
                                n_thang=('delta', lambda s: int((s > 0).sum())))
           .to_string(float_format=lambda v: f'{v:+.4f}'))
    print(f'\ntong the: {F.delta.mean():+.4f} | {int((F.delta>0).sum())}/{len(F)} nhan letterbox thang')
    print(f'''
DOC THE NAO:
  `ngoai vi` duong RO va `chi tiet` am  -> dung gia thuyet FOV, ket luan chac.
  ca hai gan 0 hoac dau lung tung        -> FOV khong phai yeu to quyet dinh o do phan giai nay.
  Nhan co n_pos_val < 50 co phuong sai rat lon — dung doc chenh lech nho o do.''')
    F.to_csv(os.path.join(RES_DIR, 'fov_crop_vs_letterbox.csv'), index=False)
else:
    print(f'cxr_only.csv thieu dong: {sorted(_need - _have)}')
    print(f'  co san: {sorted(_have)}')
    if not {'crop518', 'lb518'} <= set(ARMS):
        print('  -> thieu ca embedding -> chay 31 cho arm con thieu')
    else:
        print('  -> embedding co du, chi thieu ket qua -> CHAY LAI C3.')
        print('     C3 fit to hop nao vua thieu dong CSV vua thieu p_*.npy.')

## C5-bis — Phép cô lập biến ĐỆM: `nosq518` vs `lb518`

Hai arm này **cùng cạnh dài 518, cùng FOV 100%**, chỉ khác **~18% đệm → 0%**. Nên chênh lệch giữa
chúng không thể quy cho độ phân giải hay tầm nhìn — chỉ còn dải đen.

Dự đoán ghi trước (từ `33` §10.5, phân rã `lb518 − crop518`):

```
                     patch       cls
head               -0.0036    -0.0098
medium             +0.0108    -0.0330
tail               +0.0194    -0.0327
```

`cls` âm ở **cả** head lẫn tail — tổn hại không phụ thuộc tần suất nhãn, đúng chữ ký của nguyên nhân
cơ học. `patch` thì phụ thuộc tần suất rõ rệt, tức một cơ chế khác.

**Đọc kết quả:**

- `cls` phục hồi ≈ `+0.0098` head / `+0.0327` tail → **dải đen là thủ phạm**, xong, khỏi chạy `lb1022`
- `cls` vẫn thấp như `lb518` → không phải dải đen; `lb1022` (2.3–3.3 h) mới trả lời được
- `patch` giữ được mức lợi ở tail trong cả hai kịch bản — nó là hiệu ứng FOV, độc lập với đệm

In [ ]:
if {'nosq518', 'lb518'} <= set(ARMS):
    _ap = lambda a, k: R[R.arm.eq(a) & R.kind.eq(k)][apc].iloc[0].to_numpy('float64')
    _lbl = [c[3:] for c in apc]
    _tier = np.array([TIER_OF[l] for l in _lbl])
    # Phan ra tu C6 cua 33: FOV (lb518 - crop518) tren tung tier, tung bieu dien.
    # `cls` am gan deu o medium/tail (-0.033) va nhe hon o head (-0.0098) -> khong phu
    # thuoc tan suat nhan, dung chu ky cua nguyen nhan CO HOC (dai den).
    DUKIEN = {('patch','head'): -0.0036, ('patch','medium'): +0.0108, ('patch','tail'): +0.0194,
              ('cls','head'):   -0.0098, ('cls','medium'):   -0.0330, ('cls','tail'):   -0.0327}

    print('=== CO LAP BIEN DEM: nosq518 - lb518 (cung 518px, cung FOV, dem 18% -> 0%) ===')
    OUT = []
    for k in ('cls', 'patch'):
        if not {(a, k) for a in ('nosq518','lb518','crop518')} <= set(zip(R.arm, R.kind)): continue
        nq, lb, cr = _ap('nosq518', k), _ap('lb518', k), _ap('crop518', k)
        for t in ('head', 'medium', 'tail'):
            m = _tier == t
            OUT.append(dict(kind=k, tier=t, n=int(m.sum()),
                            nosq_tru_lb=(nq-lb)[m].mean(), nosq_tru_crop=(nq-cr)[m].mean(),
                            lb_tru_crop=(lb-cr)[m].mean(),
                            can_bu=-DUKIEN.get((k, t), np.nan)))
    Z = pd.DataFrame(OUT)
    Z['bu_duoc'] = Z.nosq_tru_lb / Z.can_bu          # 1.0 = phuc hoi dung phan da mat
    print(Z.to_string(index=False, float_format=lambda v: f'{v:+.4f}'))

    _r = Z[Z.kind.eq('cls')].bu_duoc.mean()
    print(f'''
`lb_tru_crop` phai khop bang du kien o tren — neu khong thi doc lai truoc khi ket luan.
`bu_duoc` = phan cls lay lai duoc chia cho phan da mat vi letterbox.

  bu_duoc trung binh (cls, ca 3 tier) = {_r:+.2f}''')
    if _r >= 0.6:
        print('''  >= 0.6  -> DAI DEN la thu pham chinh. Chuyen han sang nosq518,
           KHONG can chay lb1022. Cap nhat 28c/31 lam chuan cho moi arm sau.''')
    elif _r <= 0.2:
        print('''  <= 0.2  -> KHONG phai dai den. Gia thuyet con lai la do phan giai
           -> chay lb1022 (2.3-3.3h GPU).''')
    else:
        print('''  0.2-0.6 -> dai den giai thich duoc mot phan. Ca hai co che cung gop
           -> nosq1022 (giu ti le O 1022) moi la arm dung, nhung dat.''')

    # bang tung nhan, de nhin nhan nao huong loi
    for k in ('cls', 'patch'):
        if (('nosq518', k) not in set(zip(R.arm, R.kind))): continue
        Tn = pd.DataFrame(dict(label=_lbl, lb518=_ap('lb518', k), nosq518=_ap('nosq518', k)))
        Tn['delta'] = Tn.nosq518 - Tn.lb518
        Tn['tier'] = Tn.label.map(TIER_OF)
        Tn['n_pos'] = [int(YM[VA][:, LAB40.index(l)].sum()) for l in Tn.label]
        print(f'\n--- nosq518 - lb518, bieu dien `{k}` ---')
        print(Tn.sort_values('delta', ascending=False)
                .to_string(index=False, float_format=lambda v: f'{v:.4f}'))
        Tn.to_csv(os.path.join(RES_DIR, f'nosq_vs_lb_{k}.csv'), index=False)
    Z.to_csv(os.path.join(RES_DIR, 'padding_isolation.csv'), index=False)
else:
    print('can ca nosq518 va lb518 -> chay 31 voi ARMS chua nosq518 truoc')

## C6 · EHR + CXR — ba mốc gốc cạnh nhau

In [ ]:
P_EHR = os.path.join(OUT_DIR, 'cxrlt_ehr.parquet')
have_ehr = os.path.exists(P_EHR) and os.path.exists(os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json'))
if not have_ehr:
    print('chua co cxrlt_ehr.parquet -> bo qua C6, chay 29b truoc')
else:
    SPEC = json.load(open(os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json')))
    E = pd.read_parquet(P_EHR)
    cols = [c for c in SPEC['Q_FULL'] if c in E.columns]
    # EHR o muc STUDY -> phat cho tung ANH theo study_id
    EH = (E.set_index('study_id')[cols]
            .reindex(ORDER.study_id.to_numpy())
            .apply(pd.to_numeric, errors='coerce').to_numpy('float32'))
    EH = np.nan_to_num(EH, nan=0.0)          # probe tuyen tinh khong nhan NaN; mask da mang tin hieu
    print(f'EHR: {EH.shape[1]} cot, phat cho {len(EH):,} anh')

    arm, kind = best.arm, best.kind
    X = feats(arm, kind)
    combos = {'CXR-only': X, 'EHR-only': EH, 'EHR+CXR': np.concatenate([X, EH], 1)}
    rows = []
    for nm, Xc in combos.items():
        p = fit_probe(Xc[TR], YM[TR], Xc[VA])
        ap_i, ap_s = score(p)
        rows.append(dict(mo_hinh=nm, mAP_anh=ap_i.mean(), mAP_study=ap_s.mean(),
                         **{f'{t}': ap_i[[LAB40.index(l) for l in INV['tier'][t]]].mean()
                            for t in ('head', 'medium', 'tail')}))
        np.save(os.path.join(RES_DIR, f'p_fusion_{nm.replace("+","_")}.npy'), p.astype('float32'))
        print(f'  {nm:<9} mAP anh {ap_i.mean():.4f} | study {ap_s.mean():.4f}')
    del X, combos
    FU = pd.DataFrame(rows)
    print(f'\n=== BA MOC GOC (arm {arm} / {kind}) ===')
    print(FU.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
    _g = FU.set_index('mo_hinh')
    print(f'\n  EHR them vao CXR: {_g.loc["EHR+CXR","mAP_anh"] - _g.loc["CXR-only","mAP_anh"]:+.4f}')
    print(f'  CXR them vao EHR: {_g.loc["EHR+CXR","mAP_anh"] - _g.loc["EHR-only","mAP_anh"]:+.4f}')
    FU.to_csv(os.path.join(RES_DIR, 'three_baselines.csv'), index=False)

## C7 · Tổng kết

In [ ]:
print(f'''
================================================================
STAGE 3 XONG -> {RES_DIR}

  cxr_only.csv              mAP moi (arm, bieu dien), ca hai muc + tung nhan
  fov_crop_vs_letterbox.csv chenh lech tung nhan giua hai arm
  three_baselines.csv       EHR-only / CXR-only / EHR+CXR
  p_*.npy                   xac suat tren val, de dung lai o Stage 4

BON CAU DA TRA LOI
  1. FOV     -> xem C5, doc theo NHOM GIA THUYET chu khong chi macro
  2. bieu dien -> xem C4; `31` do cos(cls,patch) ~ 0.1 nen `both` co the hon ca hai
  3. muc anh vs study -> xem bang hieu o C4
  4. ba moc goc -> xem C6

NHO
  Day la probe DONG CUNG. Vo dich 0.281 mAP dung fine-tune + ensemble da do phan giai.
  Chenh lech duoi 0.005 la nhieu (hang 2-4 cua challenge cach nhau 0.004).

TIEP THEO — chi lam thu da CO BANG CHUNG o tren
  lb1022 (~2.3-3h GPU)  chi neu lb518 thang crop518 ro ret
  ASL thay BCE          5/9 doi top dung weighted ASL; day la don bay chac nhat
  fine-tune block 9-11  don bay lon nhat, nhung khong con la "RAD-DINO mac dinh"
================================================================''')